# 12. Looped Drifting

Everything so far uses a drifting model the way the paper does: **one network evaluation, no time variable**. That is the
selling point, and it is also a ceiling. A diffusion or flow model can be given more compute at test time (more steps, a finer
grid, more seeds) and gets better; a one-step generator cannot. A sibling tutorial on diffusion (`learning-diffusion`,
its notebook 8 and `THEORY.md` §12) studies a way to give *flows* that knob: **looped** models, in which one tied network is applied
repeatedly to a carried state, trained with a loss per step and no backpropagation through time (the cited works are Looped Flows,
Flow Reasoning Models and Denoising Recursion Models; I have used their summary there, not the papers). This notebook asks what carries
over to drifting, and every answer is a measurement.

The questions, in the order we answer them:

1. **Does a tied loop buy anything?** Compare, on the same drifting loss, one pass of one block, one pass of $K$ untied blocks
   (same FLOPs, $K\times$ the parameters), and $K$ loops of the one block. (§3)
2. **How should the loop be supervised?** The looped-flow recipe puts a loss on every iterate. A drifting loss is not a per-step
   denoising target, so this is not obviously right. We ablate it, along with the stop-gradient between iterates and shared noise. (§3)
3. **Does the loop converge, and does it extrapolate?** Run it past its training depth. Add the diffusion literature's two remedies:
   training on detached closed-loop carries (the idea behind Fixed-Point Forcing), and self-conditioning on the sample itself. (§4)
4. **Is stability a verifier?** If a loop settles, is the settled sample good? Compare the loop residual with a score drifting
   already owns: the field norm $\|\mathbf{V}^{+}\|$ (Notebook 0a, "Tweedie's denoiser"). (§5)
5. **What does overgenerate-score-filter buy, and what does it cost?** (§6)
6. **Given a fixed number of network calls, do you spend them on depth or on seeds?** (§7)

One accounting point up front, because it changes what "1-NFE" means. A loop of depth $k$ costs $k$ block evaluations. Looped drifting
therefore **gives up the 1-NFE property**; it is only 1-NFE in *parameters*. Whether that trade is worth making is what §3 and §7 measure.
Theory is in `THEORY.md` §16.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print("device:", device)

## 1. The toy, and what "good" means

The target is the spiral of Notebooks 2 and 3. Because it is a 2-D set we can compute, for any sample, a **ground-truth quality
label** that no real screening workload has: a sample is *good* if it lies within $0.06$ of a 50,000-point pool of the data
distribution (the data's own noise has standard deviation $\approx0.03$ per axis, and a fresh batch of data is good 100% of the time).
We also track the 24×24 total variation to a reference set (the floor for two independent draws is printed below) and **coverage**: the fraction of
the reference's high-mass histogram bins that contain at least one sample, which is the number that a filter can quietly destroy.

`BANK` is a stored set of 512 positives. A screening pipeline has one (picodrift's distilled positive bank is exactly this); §5
uses it as a scorer, never as a label.

In [ ]:
NEG_INF = -1e6

def compute_V(x, y_pos, y_neg=None, tau=0.2):
    """Algorithm-2 field: Laplace kernel, joint row+column softmax; y_neg=None means self-negatives, diagonal masked."""
    N, P = x.shape[0], y_pos.shape[0]
    self_neg = y_neg is None
    y_neg = x if self_neg else y_neg
    d_neg = torch.cdist(x, y_neg)
    if self_neg: d_neg = d_neg - NEG_INF * torch.eye(N)          # -1e6 on the logit diagonal
    logit = torch.cat([-torch.cdist(x, y_pos) / tau, -d_neg / tau], 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    A_pos, A_neg = A[:, :P], A[:, P:]
    return (A_pos * A_neg.sum(1, keepdim=True)) @ y_pos - (A_neg * A_pos.sum(1, keepdim=True)) @ y_neg

def normalised_drift(V): return V / V.pow(2).mean().sqrt().clamp_min(1e-8)

def drift_loss(x, y_pos, taus=(0.05, 0.2, 0.8), eta=0.05):
    with torch.no_grad():                                         # per-tau normalisation, then sum
        V = eta * sum(normalised_drift(compute_V(x, y_pos, None, tau=t)) for t in taus)
    return F.mse_loss(x, (x + V).detach())

def spiral(n):
    t = torch.sqrt(torch.rand(n)) * 3.5 * math.pi
    r = 0.35 * t
    return (torch.stack([r * t.cos(), r * t.sin()], -1) + 0.12 * torch.randn(n, 2)) / 3.9

def tv2d(A, B, bins=24, lim=1.3):
    h = lambda z: (lambda H: H / H.sum())(torch.histogramdd(z.cpu(), bins=[bins, bins], range=[-lim, lim, -lim, lim])[0])
    return 0.5 * (h(A) - h(B)).abs().sum().item()

torch.manual_seed(123); POOL = spiral(50000)                      # fine reference set: ground-truth "is this sample good?"
torch.manual_seed(1);   REF = spiral(20000); BANK = spiral(512)   # TV reference, and the stored positives a screen would have

@torch.no_grad()
def nn_dist(x): return torch.cdist(x, POOL).min(1).values
def good(x, thr=0.06): return nn_dist(x) < thr                   # within 0.06 of the data manifold

_h = lambda z: torch.histogramdd(z.cpu(), bins=[24, 24], range=[-1.3, 1.3, -1.3, 1.3])[0]
DENSE = (_h(REF) / _h(REF).sum() > 1 / 1500)                      # bins that carry real mass
def coverage(z): return ((_h(z) > 0) & DENSE).sum().item() / DENSE.sum().item()

torch.manual_seed(2)
print(f"data: good fraction {good(spiral(20000)).float().mean():.3f}   TV floor {tv2d(spiral(20000), REF):.3f}   coverage {coverage(spiral(4000)):.2f}")

## 2. A looped generator

The block maps $(\boldsymbol{\epsilon}, \mathbf{z}_k)\mapsto\mathbf{z}_{k+1}=\mathbf{z}_k+f_\theta(\boldsymbol{\epsilon},\mathbf{z}_k)$, with $\mathbf{z}_0=\mathbf{0}$
and a linear readout $\mathbf{x}_k=W\mathbf{z}_k$. The same noise $\boldsymbol{\epsilon}$ enters at every iterate, so $\mathbf{x}_1,\mathbf{x}_2,\dots$ is one
trajectory for one sample, and **every iterate is a candidate output**, which is what makes a depth knob possible.

Three comparators, all built from the same block:

* `Looped` at depth 1 — the equal-*parameter* one-pass baseline.
* `Untied(K)` — $K$ separate blocks in one pass: the equal-*FLOPs* one-pass baseline, with $K\times$ the parameters.
* `SelfCond` — the carried state *is* the sample, $\mathbf{x}_{k+1}=\mathbf{x}_k+f_\theta(\boldsymbol{\epsilon},\mathbf{x}_k)$, the drifting analogue of self-conditioning on the previous clean prediction.

In [ ]:
def mlp(i, h, o, nl):
    L, d = [], i
    for _ in range(nl): L += [nn.Linear(d, h), nn.SiLU()]; d = h
    return nn.Sequential(*L, nn.Linear(d, o))

class Looped(nn.Module):
    """One tied block applied K times to a carried latent z. Every iterate is read out as a 2-D sample."""
    def __init__(s, h=64, dz=32, de=8, nl=2, fresh=False):
        super().__init__(); s.de, s.dz, s.fresh = de, dz, fresh
        s.block = mlp(de + dz, h, dz, nl); s.read = nn.Linear(dz, 2)
    def step(s, eps, z): return z + s.block(torch.cat([eps, z], -1))
    def rollout(s, eps, K, detach=True, z=None):
        z = torch.zeros(eps.shape[0], s.dz) if z is None else z
        out = []
        for _ in range(K):
            e = torch.randn_like(eps) if s.fresh else eps       # fresh=True breaks the shared trajectory
            z = s.step(e, z.detach() if detach else z); out.append((s.read(z), z))
        return out

class SelfCond(Looped):
    """The carried state is the sample itself: x_{k+1} = x_k + f(eps, x_k)  (self-conditioning)."""
    def __init__(s, h=64, de=8, nl=2):
        super().__init__(h=h, dz=2, de=de, nl=nl); s.read = nn.Identity()

class Untied(nn.Module):
    """Same FLOPs as Looped(K), but K separate blocks: K times the parameters, one pass."""
    def __init__(s, K, h=64, dz=32, de=8, nl=2):
        super().__init__(); s.de, s.dz = de, dz
        s.blocks = nn.ModuleList([mlp(de + dz, h, dz, nl) for _ in range(K)]); s.read = nn.Linear(dz, 2)
    def rollout(s, eps, K=None, **kw):
        z = torch.zeros(eps.shape[0], s.dz)
        for b in s.blocks: z = z + b(torch.cat([eps, z], -1))
        return [(s.read(z), z)]

nparams = lambda m: sum(p.numel() for p in m.parameters())
print(f"Looped block: {nparams(Looped()):,} params   Untied x4: {nparams(Untied(4)):,} params   SelfCond block: {nparams(SelfCond()):,} params")

## 3. Training a loop with a drifting loss

The drifting loss at an iterate $\mathbf{x}_k$ is the usual one: regress $\mathbf{x}_k$ onto $\texttt{sg}(\mathbf{x}_k+\mathbf{V}(\mathbf{x}_k))$, with the
negatives being the batch's other samples *at the same iterate*. The design space is then three binary choices, all exposed in `fit`:

* **which iterates carry a loss** — every one (`mode="all"`, the looped-flow recipe) or only the last (`"last"`);
* **whether the state is detached between iterates** — stop-gradient (cheap, no BPTT) or full backpropagation through the loop;
* **whether noise is shared** across iterates (a trajectory) or fresh at each (`fresh`).

`rand_depth=J` is a fourth option for §4: roll the loop for a random $j<J$ steps under `no_grad`, then supervise one step from that detached carry.
That trains the block on states produced by its own closed-loop rollout, which is the exposure-bias fix of Fixed-Point Forcing in spirit (not a reproduction of it).

In [ ]:
def fit(make, steps=5000, K=4, mode="all", detach=True, rand_depth=None, N=256, seed=0, lr=2e-3):
    """mode: 'all' = drifting loss on every iterate, 'last' = final iterate only.
       detach: stop-gradient on the carried state between iterates (False = full BPTT).
       rand_depth=J: carry-in from a detached closed-loop rollout of random length j < J, then one supervised step."""
    torch.manual_seed(seed); m = make()
    opt = torch.optim.Adam(m.parameters(), lr); sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for _ in range(steps):
        y, eps = spiral(N), torch.randn(N, m.de)
        if rand_depth:
            j = int(torch.randint(0, rand_depth, ()))
            with torch.no_grad(): z = m.rollout(eps, j)[-1][1] if j else None
            outs = m.rollout(eps, 1, z=z)
        else:
            outs = m.rollout(eps, K, detach=detach)
        used = outs if mode == "all" else outs[-1:]
        loss = sum(drift_loss(x, y) for x, _ in used) / len(used)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 2.0); opt.step(); sch.step()
    return m

@torch.no_grad()
def rollout_x(m, k, n=8000, seed=5):
    torch.manual_seed(seed)
    return [x for x, _ in m.rollout(torch.randn(n, m.de), k)]

Seven arms at depth $K=4$, 5000 steps each. The four that carry the argument run three seeds; the three ablations run one, so
read their rows as direction, not size.

In [ ]:
K = 4
arms = {                                   # name: (factory, fit kwargs, seeds)
    "1 pass, 1 block (equal params)": (lambda: Looped(),        dict(K=1),                         (0, 1, 2)),
    "1 pass, 4 blocks (equal FLOPs)": (lambda: Untied(K),       dict(),                            (0, 1, 2)),
    "loop, sg between, loss all":     (lambda: Looped(),        dict(K=K, mode="all"),             (0, 1, 2)),
    "loop, BPTT, loss last":          (lambda: Looped(),        dict(K=K, mode="last", detach=False), (0, 1, 2)),
    "loop, sg between, loss last":    (lambda: Looped(),        dict(K=K, mode="last"),            (0,)),
    "loop, BPTT, loss all":           (lambda: Looped(),        dict(K=K, mode="all", detach=False), (0,)),
    "loop, fresh noise per iterate":  (lambda: Looped(fresh=True), dict(K=K, mode="all"),          (0,)),
}
models, t0 = {}, time.time()
print(f"{'arm':32s} {'TV':>14s} {'good frac':>14s}   seeds")
for name, (make, kw, seeds) in arms.items():
    r = []
    for s in seeds:
        m = fit(make, seed=s, **kw); x = rollout_x(m, kw.get("K", K))[-1]
        r.append((tv2d(x, REF), good(x).float().mean().item())); models.setdefault(name, m)
    a = np.array(r)
    print(f"{name:32s} {a[:,0].mean():6.3f} ± {a[:,0].std():.3f} {a[:,1].mean():8.3f} ± {a[:,1].std():.3f}   n={len(seeds)}   ({time.time()-t0:.0f}s)")

**Looping buys depth, not magic.** At equal parameters a single pass is poor (TV 0.428), and four loops of the same block with BPTT and a loss on the last iterate
reach TV 0.156 ± 0.018. The equal-FLOPs untied network reaches 0.181 ± 0.013 with four times the parameters (35,522 against 8,930).
So the tied loop is *at least as good* as spending the same FLOPs on untied depth, at a quarter of the parameter count. It is not better
per FLOP, and it is not 1-NFE: that was never on offer.

**The looped-flow supervision recipe does not transfer.** Putting the drifting loss on every iterate with a stop-gradient between them,
the recipe that works for looped flows, is much worse here: 0.361 ± 0.068 against 0.156. The single-seed ablations point the same way:
loss on all iterates is poor with BPTT too (0.366), and the stop-gradient costs a lot even with a last-iterate loss (0.293 against 0.156, three seeds for the latter). My reading, which these
rows are consistent with but do not isolate: a flow step has its own target (denoise from $t_i$ to $t_{i+1}$), so intermediate losses are compatible
with the final one. A drifting loss at iterate $k$ instead says *this iterate is already a sample from $p$*, which asks the early iterates to do the whole job
and leaves nothing for the loop to refine. The shared-noise ablation (fresh noise per iterate: 0.380) does not separate from the shared-noise arm
(0.361, seed sd 0.068), so the "temporal alignment" ingredient of Looped Flows is neither supported nor refuted here.

## 4. Does the loop converge, and does it extrapolate?

A model trained at depth 4 is asked for depth $k$ up to 32. Five models: the two from §3, a model trained on detached closed-loop carries of random length
$<8$ and one with $<16$, and a self-conditioning model on the same random-depth schedule as the first. Each cell is TV / good fraction at that depth. The second table is the mean step $\|\mathbf{x}_{k+1}-\mathbf{x}_k\|$:
at a fixed point it is zero.

In [ ]:
ks = (1, 2, 4, 6, 8, 12, 16, 24, 32)
stack = {
    "loop K=4, BPTT, last":     models["loop, BPTT, loss last"],
    "loop K=4, sg, all":        models["loop, sg between, loss all"],
    "random depth <8 (FPF-like)":  fit(lambda: Looped(), rand_depth=8),
    "random depth <16":            fit(lambda: Looped(), rand_depth=16),
    "self-conditioning, depth <8": fit(lambda: SelfCond(), rand_depth=8),
}
print(f"{'TV / good fraction at loop depth k':32s}" + "".join(f"{'k='+str(k):>11s}" for k in ks))
curves = {}
for name, m in stack.items():
    xs = rollout_x(m, max(ks)); curves[name] = [(tv2d(xs[k-1], REF), good(xs[k-1]).float().mean().item()) for k in ks]
    print(f"{name:32s}" + "".join(f"{tv:6.2f}/{g:.2f}" for tv, g in curves[name]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for name, c in curves.items():
    ax[0].plot(ks, [t for t, _ in c], "o-", ms=3, label=name)
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("loop depth k at inference"); ax[0].set_ylabel("TV to data"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
m = stack["random depth <8 (FPF-like)"]; xs = rollout_x(m, 32, n=3000)
for k, c in zip((2, 6, 8, 16, 32), plt.cm.viridis(np.linspace(0, 0.9, 5))):
    ax[1].scatter(*xs[k-1].cpu().T, s=1.5, color=c, label=f"k={k}", alpha=0.5)
ax[1].legend(fontsize=7, markerscale=5); ax[1].set_aspect("equal"); ax[1].set_title("random-depth model, samples at depth k", fontsize=9)
plt.tight_layout(); plt.show()

with torch.no_grad():
    print("\nmean step size |x_{k+1} - x_k|  (a fixed point has this = 0)")
    print(f"{'':32s}" + "".join(f"{'k='+str(k):>9s}" for k in (2, 4, 8, 16, 32)))
    for name, m in stack.items():
        xs = rollout_x(m, 33); print(f"{name:32s}" + "".join(f"{(xs[k]-xs[k-1]).norm(dim=-1).mean().item():9.4f}" for k in (2, 4, 8, 16, 32)))

**There is no fixed point, and the usable depth is the depth you trained.**

* The BPTT model is good at exactly its training depth (TV 0.18 at $k=4$) and bad everywhere else (0.76 at $k=1$, 0.70 at $k=6$). It is a four-layer network that happens to share weights, not an iteration that settles; its step size stays near 0.19 at every depth.
* The stop-gradient, all-iterates model degrades past its horizon: TV 0.38 at $k=4$, 0.62 at $k=12$, 0.75 at $k=32$.
* Training on random-depth detached carries moves the horizon with the training depth: the $<8$ model is best at $k=4$ to $6$ (0.29, 0.30) and collapses by $k=24$ (0.74); the $<16$ model is best at $k=12$ (0.28), still fine at 16 (0.33), and fails at 24 (0.54). This is the exposure-bias story, and the remedy
  widens the range. It does not turn the loop into a contraction: nothing in the loss asks $\mathbf{x}_{k+1}=\mathbf{x}_k$, and the step size never approaches zero.
* **Self-conditioning does converge**, exactly: its step size is 0.0131 at $k=2$, 0.0005 at $k=4$ and 0.0000 from $k=8$, and TV freezes at 0.40. It converges to a *worse* answer than the latent loop reached (0.29), with 80% of samples good. That is the spurious-attractor failure of the
  diffusion literature, reproduced: stability is not correctness. (Step size does not see the collapse of the other models either: the $<8$ latent model's mean step is 0.0375 at $k=8$ and 0.0465 at $k=32$, while its TV goes from 0.34 to 0.76.)

For drifting the reason is structural. The fixed point of *training* is $\mathbf{V}=\mathbf{0}$ in distribution (`THEORY.md` §4.1), a statement about the pushforward $q_k$, not about any one sample's trajectory. The loss is satisfied by an iterate that moves each sample anywhere
as long as the *population* is right.

## 5. Stability as a verifier

If the loop has settled, the sample should be good; the flow-reasoning literature uses the step-to-step residual as a self-check. We score each sample three ways and ask how well the score separates good from bad (AUROC: the probability a good sample outscores a bad one; 0.5 is chance):

* the **loop residual** $\|\mathbf{x}_{K+1}-\mathbf{x}_K\|$, and the longer-gap $\|\mathbf{x}_{K+4}-\mathbf{x}_K\|$;
* the **field norm** $\|\mathbf{V}^{+}(\mathbf{x})\|$ against the 512-point bank. By Notebook 0a's Tweedie check (`THEORY.md` §7.1) this is the distance a denoising step toward the data would move $\mathbf{x}$: the diffusion-style residual, but for a one-step model and one that needs no loop;
* **nearest-neighbour distance to the bank**, shown only as a ceiling: it is close to the label's own definition, so its AUROC is partly by construction.

In [ ]:
def auroc(score, ok):                      # P(a good sample outscores a bad one)
    r = score.argsort().argsort().float() + 1; n1 = ok.sum(); n0 = (~ok).sum()
    return ((r[ok].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)).item()

def vplus_norm(x, bank=BANK, tau=0.05):    # |V+(x)|: how far a Tweedie/denoising step toward the data would move x
    return ((-torch.cdist(x, bank) / tau).softmax(-1) @ bank - x).norm(dim=-1)

def scores(m, K, n=8000, seed=5):
    xs = rollout_x(m, K + 4, n, seed); x = xs[K-1]
    return x, {"loop residual |x_K+1 - x_K|": (xs[K] - x).norm(dim=-1), "loop residual |x_K+4 - x_K|": (xs[K+3] - x).norm(dim=-1),
               "field norm |V+| vs bank": vplus_norm(x), "NN distance to bank": torch.cdist(x, BANK).min(1).values,
               "random": torch.rand(n)}

for name, K_ in (("random depth <8 (FPF-like)", 6), ("loop K=4, BPTT, last", 4), ("loop K=4, sg, all", 3)):
    x, sc = scores(stack[name], K_); ok = good(x)
    print(f"{name}, scored at depth {K_}: good fraction {ok.float().mean():.3f}")
    for k, s in sc.items(): print(f"   {k:30s} AUROC {auroc(-s, ok):.3f}")

**The loop residual is a weak verifier; the bank's field norm is a strong one.** Residual AUROC lies between 0.50 and 0.71 over six measurements
(0.683 and 0.712 for the random-depth model), and is at chance for the stop-gradient model's one-step gap (0.497). The field norm against a 512-point bank reaches
0.925, 0.956 and 0.912, and nearest-neighbour distance 0.995–0.998. The `random` rows (0.49–0.51) confirm the metric is calibrated.

Two cautions. The high bank AUROCs are partly circular (the label and the score are both nearest-data distances; what it demonstrates is that a *small* stored bank is enough). And the bank-based score needs data at screening time. That is available for a conditional model with a positive
bank (picodrift's distilled set), and absent for a genuinely unconditional one. Convergence of a loop, by contrast, costs nothing but certifies little here: §4 showed it does not track quality.

## 6. Overgenerate, score, filter

Take 8000 samples from the random-depth model at depth 6, keep the best fraction by a score, and measure what is left. The `random` rows are the control for
"fewer samples": any change there is just subsampling noise.

In [ ]:
m, K_ = stack["random depth <8 (FPF-like)"], 6
x, sc = scores(m, K_); torch.manual_seed(9); rnd = torch.randperm(len(x))
print(f"{'score':30s} {'keep':>5s} {'good':>6s} {'TV':>6s} {'coverage':>9s}")
print(f"{'(no filter)':30s} {'100%':>5s} {good(x).float().mean():6.3f} {tv2d(x, REF):6.3f} {coverage(x):9.2f}")
for k in ("loop residual |x_K+1 - x_K|", "field norm |V+| vs bank", "random"):
    for frac in (0.5, 0.25, 0.1):
        keep = sc[k].argsort()[: int(frac * len(x))] if k != "random" else rnd[: int(frac * len(x))]
        print(f"{k:30s} {frac:5.0%} {good(x[keep]).float().mean():6.3f} {tv2d(x[keep], REF):6.3f} {coverage(x[keep]):9.2f}")

**Filtering raises validity and spends coverage, and there is no free lunch in the middle.** Unfiltered, 88.3% of samples are good, TV is 0.300 and 99% of the high-mass bins are covered. Keeping the best half by field norm lifts the good fraction to 98.7% and *improves* TV to 0.208, because
most of the unfiltered TV is off-manifold mass that the filter removes. But coverage is already down to 0.88, and further filtering reverses the TV gain: keeping 25% gives TV 0.290 with coverage 0.63, and keeping 10% gives TV 0.373 with coverage 0.54,
even though 99.5% of what remains is good. The filtered set is a sharper, narrower distribution than the model's. The loop-residual filter is worse than the field-norm filter on validity and on TV at every kept fraction, and keeping the best half *worsens* TV (0.384 against 0.300), because its score correlates with quality only weakly (§5) and discards good samples. The random-subsample rows stay at TV 0.305–0.332
and coverage 0.94–0.99, so the changes above are the filter's doing.

This is the learning-diffusion repository's EBM-filtering caveat, measured for drifting: filtering removes bad samples and cannot add missing ones, and a score that is *only* a nearest-data distance prefers the dense core of the data and strips its tails.

## 7. Spending a fixed budget: depth or seeds

Fix 24 block calls per delivered sample and split them between loop depth $K$ and number of seeds $N$ (with $KN=24$), then keep the best of the $N$ by score. This is the looped-flow papers' two test-time knobs (a finer grid, multiple noise seeds) with a drifting model. One model, one run per row, so treat differences below about 0.03 in TV as noise.

In [ ]:
slots, B = 4000, 24
print(f"best-of-N by score, K loops x N seeds = {B} block calls per delivered sample (random-depth model)")
print(f"{'score':26s} {'K':>3s} {'N':>3s} {'good':>6s} {'TV':>6s} {'coverage':>9s}")
m = stack["random depth <8 (FPF-like)"]
for sname in ("field norm |V+|", "loop residual"):
    for K_, N_ in ((1, 24), (2, 12), (3, 8), (4, 6), (6, 4), (8, 3), (12, 2), (24, 1)):
        torch.manual_seed(11); e = torch.randn(slots * N_, m.de)
        with torch.no_grad():
            xs = [x for x, _ in m.rollout(e, K_ + 1)]; x = xs[K_-1]
            s = vplus_norm(x) if sname.startswith("field") else (xs[K_] - x).norm(dim=-1)
            pick = s.view(slots, N_).argmin(1); xsel = x.view(slots, N_, 2)[torch.arange(slots), pick]
        print(f"{sname:26s} {K_:3d} {N_:3d} {good(xsel).float().mean():6.3f} {tv2d(xsel, REF):6.3f} {coverage(xsel):9.2f}")

**With a good scorer the optimum is interior, at depth near the edge of the model's horizon.** Scored by field norm, all-seeds ($K=1$, $N=24$) gives TV 0.298 and coverage 0.68: best-of-24 over a poor sampler sheds coverage. All-depth ($K=24$, $N=1$) is past the $<8$ model's horizon and collapses (TV 0.727, good 0.447). The best rows are $K=8$, $N=3$ (TV 0.198, coverage 0.98) and $K=6$, $N=4$ (0.218, 0.93).
With the loop residual as the scorer there is no such optimum: its best TV is 0.365 ($K=8$) and the all-seeds row is 0.631 with coverage 0.55, since a weak scorer makes extra seeds expensive.

The usable lesson is conditional on §5: **depth and seeds are complements only if there is a scorer that works.** Without one, extra seeds buy nothing and extra depth is limited by the horizon of §4.

## What this notebook established

* A tied loop is a *parameter-efficient* way to buy depth: TV 0.156 ± 0.018 at 8,930 parameters against 0.181 ± 0.013 for an untied net of the same FLOPs at 35,522. It is not 1-NFE, and it is not better per FLOP.
* The looped-flow recipe (a loss on every iterate) is **bad** for drifting: 0.361 ± 0.068 against 0.156 for BPTT and a last-iterate loss. A drifting loss says each iterate is already a sample, so supervising the early ones wastes the loop.
* A drifting loop does not converge to a fixed point by itself, and it does not extrapolate: good depth equals trained depth (0.28–0.30 TV inside the horizon, 0.7+ well past it). Detached closed-loop carries move the horizon; they do not remove it.
* Self-conditioning *does* reach an exact fixed point (step 0.0000) but at a worse answer (TV 0.40 against 0.29): stable and wrong.
* The loop residual is a weak verifier (AUROC 0.50–0.71). The bank field norm $\|\mathbf{V}^{+}\|$, the Tweedie residual, is a strong one (0.91–0.96) and works for any one-step model.
* Filtering trades coverage for validity (coverage 0.99 → 0.88 → 0.54 as the kept fraction goes 100% → 50% → 10%); best-of-$N$ split with depth beats either extreme (TV 0.198 at $K=8$, $N=3$ against 0.298 and 0.727).

Everything here is one 2-D toy, one training run per row except where marked, on CPU. In particular nothing measured here concerns Protenix; `PROTENIX_DRIFTING.md` §14 records how the verifier idea would be tested there.

Related: [11. From Notebooks to picodrift](<11. From Notebooks to picodrift.ipynb>), where the positive banks used here come from in practice, and the [README](README.md) for the tour.